# SageMaker Demo: Employee Attrition Prediction Using Feature Store and XGBoost

This notebook demonstrates how to use Amazon SageMaker's Feature Store and XGBoost built-in algorithm to predict employee attrition.

In [1]:
!pip install pandas cffi sagemaker sagemaker-feature-store-pyspark
!pip install --upgrade sagemaker

import sagemaker
print(dir(sagemaker))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [2]:
# Step 1: Setup
import sagemaker, boto3, time
#from sagemaker import get_execution_role
import pandas as pd
from time import gmtime, strftime
from sklearn.model_selection import train_test_split 

try:
    from sagemaker import get_execution_role
    role = get_execution_role()
except:
    import boto3
    role = boto3.client("sts").get_caller_identity()["Arn"]

region = boto3.Session().region_name

# S3 bucket for storing data
bucket = 'sagemaker-eu-central-1-112242697189'

# Load the dataset
file_path = 'Employee.csv'
employee_df = pd.read_csv(file_path)
employee_df.head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0
1,Bachelors,2013,Pune,1,28,Female,No,3,1
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0
3,Masters,2016,Bangalore,3,27,Male,No,5,1
4,Masters,2017,Pune,3,24,Male,Yes,2,1


In [3]:
# Step 2: Data Preparation
# Convert categorical columns to numeric
employee_df['Education'] = employee_df['Education'].astype('category').cat.codes
employee_df['City'] = employee_df['City'].astype('category').cat.codes
employee_df['Gender'] = employee_df['Gender'].astype('category').cat.codes
employee_df['EverBenched'] = employee_df['EverBenched'].map({'Yes': 1, 'No': 0})

# Drop rows with NaN values in the target column
employee_df.dropna(subset=['LeaveOrNot'])

# Convert target column to numeric if needed
employee_df['LeaveOrNot'] = employee_df['LeaveOrNot'].astype(int)

# Ensure no missing values in feature columns
employee_df = employee_df.dropna()

# Verify all columns are numeric
print(employee_df.dtypes)

# Define features and target
feature_columns = ['Education', 'JoiningYear', 'City', 'PaymentTier', 'Age','Gender', 'EverBenched', 'ExperienceInCurrentDomain']
target_column = 'LeaveOrNot'

employee_df = employee_df[[target_column] + feature_columns]

# Display the transformed dataset
employee_df.head()

Education                     int8
JoiningYear                  int64
City                          int8
PaymentTier                  int64
Age                          int64
Gender                        int8
EverBenched                  int64
ExperienceInCurrentDomain    int64
LeaveOrNot                   int64
dtype: object


,LeaveOrNot,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain
0,0,0,2017,0,3,34,1,0,0
1,1,0,2013,2,1,28,0,0,3
2,0,0,2014,1,3,38,0,0,2
3,1,1,2016,0,3,27,1,0,5
4,1,1,2017,2,3,24,1,1,2


In [4]:
# Use the existing SageMaker execution role
role = 'arn:aws:iam::112242697189:role/service-role/AmazonSageMaker-ExecutionRole-20260506T092538'

# Create boto3 clients
sagemaker_client = boto3.client('sagemaker')
featurestore_client = boto3.client('sagemaker-featurestore-runtime')

# Create feature group name
feature_group_name = 'employee-feature-group-' + strftime('%Y%m%d%H%M%S', gmtime())

# Define schema
record_identifier_name = 'EmployeeID'
event_time_feature_name = 'EventTime'

# Prepare data
employee_df[event_time_feature_name] = pd.to_datetime('now').strftime('%Y-%m-%dT%H:%M:%S.%fZ')
employee_df[record_identifier_name] = employee_df.index

# Create feature definitions from DataFrame
feature_definitions = []
for col in employee_df.columns:
    dtype = str(employee_df[col].dtype)
    if 'int' in dtype:
        feature_type = 'Integral'
    elif 'float' in dtype:
        feature_type = 'Fractional'
    else:
        feature_type = 'String'
    
    feature_definitions.append({'FeatureName': col,'FeatureType': feature_type})

# Create feature group
sagemaker_client.create_feature_group(
                                        FeatureGroupName=feature_group_name,
                                        RecordIdentifierFeatureName=record_identifier_name,
                                        EventTimeFeatureName=event_time_feature_name,
                                        FeatureDefinitions=feature_definitions,
                                        OnlineStoreConfig={'EnableOnlineStore': True},
                                        OfflineStoreConfig={'S3StorageConfig': {'S3Uri': 's3://sagemaker-eu-central-1-112242697189/features'}},
                                        RoleArn=role
                                        )


{'FeatureGroupArn': 'arn:aws:sagemaker:eu-central-1:112242697189:feature-group/employee-feature-group-20260514090649',
 'ResponseMetadata': {'RequestId': 'ecf5f5c9-9f4e-48e3-b0a0-afd8c67148ab',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'ecf5f5c9-9f4e-48e3-b0a0-afd8c67148ab',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '117',
   'date': 'Thu, 14 May 2026 09:06:50 GMT'},
  'RetryAttempts': 0}}

In [5]:
# Check the status of the Feature Group
response = sagemaker_client.describe_feature_group(FeatureGroupName=feature_group_name)
status = response.get("FeatureGroupStatus")
print(f"Feature Group Status: {status}")

# Wait for feature group to be created
while status == "Creating":
    print("Waiting for feature group to be created...")
    time.sleep(30)
    response = sagemaker_client.describe_feature_group(FeatureGroupName=feature_group_name)
    status = response.get("FeatureGroupStatus")

if status == "Created":
    print("Feature Group is Created and ready for use. Proceeding with ingestion...")
    
    # Convert DataFrame to records for ingestion
    records = []
    for _, row in employee_df.iterrows():
        record = []
        for col in employee_df.columns:
            value = str(row[col])
            record.append({
                'FeatureName': col,
                'ValueAsString': value
            })
        records.append({
            'Record': record
        })
    
    # Ingest data into the Feature Store
    featurestore_client.put_record(FeatureGroupName=feature_group_name,Record=records[0]['Record'])
    
    # For multiple records, use batch ingestion
    for record in records:
        featurestore_client.put_record(FeatureGroupName=feature_group_name,Record=record['Record'])
    print('Data ingested into Feature Store.')


Feature Group Status: Creating
Waiting for feature group to be created...


Feature Group is Created and ready for use. Proceeding with ingestion...


Data ingested into Feature Store.


In [6]:
# Define the feature group name and features you want to retrieve
feature_names = ['Education', 'JoiningYear', 'City', 'PaymentTier', 'Age', 'Gender', 'EverBenched', 'ExperienceInCurrentDomain', 'LeaveOrNot']

# Retrieve records and convert to DataFrame
records = []
for record_id in employee_df.index.astype(str):
    try:
        response = featurestore_client.get_record(FeatureGroupName=feature_group_name,RecordIdentifierValueAsString=str(record_id),FeatureNames=feature_names)
        # Check if 'Record' is in the response and add to records list
        if 'Record' in response:
            record = {feature['FeatureName']: feature['ValueAsString'] for feature in response['Record']}
            records.append(record)
        else:
            print(f"Record with ID {record_id} not found.")
    except Exception as e:
        print(f"Error retrieving record {record_id}: {e}")

# Convert the list of records to a DataFrame
retrieved_df = pd.DataFrame(records)

# Check if we have any retrieved records
if not retrieved_df.empty:
    # Split the data into training and test sets
    train_df, test_df = train_test_split(retrieved_df, test_size=0.2, random_state=42)
    print("Training and test data split after retrieval from Feature Store.")
else:
    print("No records retrieved. Please check the feature group and identifiers.")

Training and test data split after retrieval from Feature Store.


## Train the Model Using Local Data with S3 Mode (Default)

In [7]:
# Initialize S3 client
s3 = boto3.client('s3')

# Define your S3 bucket and prefix
bucket = 'sagemaker-eu-central-1-112242697189'
prefix = 'input-data'

# Save the data locally first
train_file = 'train.csv'
validation_file = 'validation.csv'
train_df.to_csv(train_file, index=False)
test_df.to_csv(validation_file, index=False)

# Upload the data to S3
s3.upload_file(train_file, bucket, f'{prefix}/train/{train_file}')
s3.upload_file(validation_file, bucket, f'{prefix}/validation/{validation_file}')

print(f"Training data uploaded to s3://{bucket}/{prefix}/train/{train_file}")
print(f"Validation data uploaded to s3://{bucket}/{prefix}/validation/{validation_file}")


Training data uploaded to s3://sagemaker-eu-central-1-112242697189/input-data/train/train.csv
Validation data uploaded to s3://sagemaker-eu-central-1-112242697189/input-data/validation/validation.csv


In [8]:
# Initialize hyperparameters
hyperparameters = {
                    "max_depth": "12",
                    "eta": "0.2",
                    "gamma": "0",
                    "min_child_weight": "6",
                    "learning_rate": "0.01",
                    "subsample": "0.7",
                    "objective": "reg:squarederror",
                    "num_round": "100",
                    "early_stopping_rounds": "25",
                    "colsample_bytree": "0.8",
                    "n_estimators": "900",
                    }

# Set paths
bucket = 'sagemaker-eu-central-1-112242697189'
prefix = 'demo-built-in-algorithm'
output_path = f's3://{bucket}/{prefix}/output'
role = 'arn:aws:iam::112242697189:role/service-role/AmazonSageMaker-ExecutionRole-20260506T092538'

# Get XGBoost container URI
region = boto3.Session().region_name
xgboost_container = f"246618743249.dkr.ecr.{region}.amazonaws.com/sagemaker-xgboost:1.7-1"

# Create SageMaker client
sagemaker_client = boto3.client('sagemaker')

# Create training job
training_job_name = f'xgboost-employee-attrition-{int(time.time())}'

response = sagemaker_client.create_training_job(
    TrainingJobName=training_job_name,
    RoleArn=role,
    AlgorithmSpecification={'TrainingImage': xgboost_container,'TrainingInputMode': 'File'},
    InputDataConfig=[
        {
            'ChannelName': 'train',
            'DataSource': {
                'S3DataSource': {
                    'S3DataType': 'S3Prefix',
                    'S3Uri': f's3://{bucket}/input-data/train/',
                    'S3DataDistributionType': 'FullyReplicated'}},'ContentType': 'csv'},
        {
            'ChannelName': 'validation',
            'DataSource': {
                'S3DataSource': {
                    'S3DataType': 'S3Prefix',
                    'S3Uri': f's3://{bucket}/input-data/validation/',
                    'S3DataDistributionType': 'FullyReplicated'}},'ContentType': 'csv'}],
    OutputDataConfig={'S3OutputPath': output_path},
    ResourceConfig={'InstanceType': 'ml.m5.xlarge','InstanceCount': 1,'VolumeSizeInGB': 5},
    StoppingCondition={'MaxRuntimeInSeconds': 3600},
    HyperParameters=hyperparameters
)

print(f"Training job started: {training_job_name}")


Training job started: xgboost-employee-attrition-1778749727
